# Fraud review system: held-out results walkthrough

> **Synthetic-data warning:** PaySim is a simulation. This notebook explains the repository's frozen results; it is not evidence of real-world model effectiveness.

I keep reusable validation, feature, model, policy, and monitoring logic in `src/fraud_system`. This notebook reads the produced artifacts instead of recreating hidden business logic in cells.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METRICS = PROJECT_ROOT / "reports" / "metrics"
metrics = json.loads((METRICS / "final_metrics.json").read_text())
metrics

## 1. Why the chronological split matters

The simulator's fraud prevalence rises sharply over time. I used a chronological split because a random split would mix those periods and conceal the shift. I keep whole simulation hours together and leave the final period untouched until I freeze the model and threshold.

In [ ]:
split = metrics["split"]
pd.Series(
    {
        "train": split["train_fraud_rate"],
        "validation": split["validation_fraud_rate"],
        "test": split["test_fraud_rate"],
    }
).mul(100).rename("fraud rate (%)").plot.bar(title="Fraud prevalence by chronological period");

## 2. Model comparison

I select the operational champion by validation policy cost and use PR-AUC to break a tie. I do not use accuracy because the non-fraud majority would dominate it.

In [ ]:
pd.read_csv(METRICS / "candidate_comparison.csv")

![Validation precision-recall curves](../reports/figures/validation_precision_recall.png)

I treat the near-perfect LightGBM result as a warning about simulator separability, not as a result I would generalize to production.

## 3. Threshold and capacity form one policy

I selected the threshold on validation data under the stated cost assumptions. I still cap eligible cases at 200 per day and rank them by expected preventable loss.

In [ ]:
pd.DataFrame(
    {
        "validation": metrics["validation_policy"],
        "test": metrics["test_policy"],
    }
).loc[["threshold", "reviewed", "precision", "recall", "capacity_binding_days", "loss_avoided"]]

![Validation cost by threshold](../reports/figures/validation_policy_cost.png)

![Test confusion matrix](../reports/figures/test_confusion_matrix.png)

## 4. Monitoring view

PSI compares each test day's score distribution with validation. It identifies change but does not prove performance deterioration.

In [ ]:
daily = pd.read_csv(METRICS / "daily_monitoring.csv").set_index("day_index")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
daily[["transaction_count", "review_count"]].plot(ax=axes[0], title="Volume and reviews")
daily[["score_psi_vs_validation"]].plot(ax=axes[1], title="Score PSI")
plt.tight_layout();

## 5. Final interpretation

This project demonstrates how I approached reproducible decision engineering: lineage, validation, point-in-time features, calibrated models, a cost matrix, capacity handling, tests, monitoring, and governance documents. It does **not** show that PaySim-derived fraud patterns transfer to real people or institutions.